<a href="https://colab.research.google.com/github/Sheriff414/my_deeplearning_model/blob/master/densenet_proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip3 install einops


import torch

import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.utils.prune as prune
import torch.optim as optim
from torch.optim import lr_scheduler
#from einops.layers.torch import Reduce
import os
from torch.utils.data import DataLoader

import seaborn as sns
from torch.utils.data import random_split
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:

from torchvision.transforms.functional import InterpolationMode

In [ ]:
data_path = "C:\Users\Ahmed Sheriffdeen\Documents\TinyImageNet\archive\tiny-imagenet-200"

In [ ]:
#hyperparameters
batch_size=64

tranform_train = transforms.Compose([ transforms.Resize(224),transforms.RandomHorizontalFlip(p=0.7), transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
tranform_test = transforms.Compose([ transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

train = torchvision.datasets.CIFAR100(root="./data", train=True, download=True, transform=tranform_train)
val_size = 10000
train_size = len(train) - val_size
train, val = random_split(train, [train_size, val_size])
test = torchvision.datasets.CIFAR100(root="./data", train=True, download=True, transform=tranform_test)

#  train, val and test datasets to the dataloader
train_loader = DataLoader(train, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val, batch_size=batch_size, shuffle=False)

100%|██████████| 169001437/169001437 [00:02<00:00, 83404910.31it/s]


Extracting ./data/cifar-100-python.tar.gz to ./data
Files already downloaded and verified


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#Teacher model DENSENET121
bmodel=torchvision.models.densenet121(pretrained=True)
#num_ftrs=bmodel.fc.in_features
bmodel.classifier=nn.Linear(in_features=1024, out_features=100, bias=True)
bmodel = bmodel.to(device)

In [ ]:
# Hyper-parameters
num_epochs = 100
learning_rate = 0.01
criterionA = nn.CrossEntropyLoss()
optimizerA = torch.optim.SGD(bmodel.parameters(), momentum= 0.9, weight_decay=0.0005, lr=learning_rate) #weight_decay=0.005
n_total_steps = len(train_loader)
scheduler = lr_scheduler.MultiStepLR(optimizerA, milestones=(30, 70), gamma=0.1)
#scheduler = lr_scheduler.StepLR(optimizerA, step_size=2, gamma=0.1)


#TRAINING BASE MODEL

for epoch in range(num_epochs): #I decided to train the model for 50 epochs
    # Decay Learning Rate
    scheduler.step()
    loss_var = 0

    for idx, (images, labels) in enumerate(train_loader):
        images = images.to(device=device)
        labels = labels.to(device=device)
        ## Forward Pass
        optimizerA.zero_grad()
        scores = bmodel(images)
        loss = criterionA(scores,labels)
        loss.backward()
        optimizerA.step()
        loss_var += loss.item()
        if idx%64==0:
            print(f'Epoch [{epoch+1}/{num_epochs}] || Step [{idx+1}/{len(train_loader)}] || Loss:{loss_var/len(train_loader)}')
    print(f"Loss at epoch {epoch+1} || {loss_var/len(train_loader)}")

    with torch.no_grad():
        correct = 0
        samples = 0
        for idx, (images, labels) in enumerate(val_loader):
            images = images.to(device=device)
            labels = labels.to(device=device)
            outputs = bmodel(images)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum()
            samples += preds.size(0)
            #acc= correct/samples*100
        print(f"accuracy {float(correct) / float(samples) * 100:.2f} percentage || Correct {correct} out of {samples} samples")
    #schedulerA.step(acc)
    PATH1 = '/home/lm/bmodel_alexnet-c100.pt'
    torch.save(bmodel.state_dict(), PATH1)
print('Finished Training')
#PATH1 = 'bmodel.pt'
#torch.save(bmodel.state_dict(), PATH1)

In [ ]:
#CREATING SUBNETS & PRUNING
#Copy1
import copy
m_copy=copy.deepcopy(bmodel)
m2_copy=copy.deepcopy(bmodel)

#Pruning1
for name, module in m_copy.named_modules():
    # prune 50% of connections in all 2D-conv layers
    if isinstance(module, torch.nn.Conv2d):
        prune.l1_unstructured(module, name='weight', amount=0.5)
    # prune 50% of connections in all linear layers
    elif isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.5)

print(dict(m_copy.named_buffers()).keys())  # to verify that all masks exist

#Pruning2
for name, module in m2_copy.named_modules():
    # prune 50% of connections in all 2D-conv layers
    if isinstance(module, torch.nn.Conv2d):
        prune.l1_unstructured(module, name='weight', amount=0.7)
    # prune 50% of connections in all linear layers
    elif isinstance(module, torch.nn.Linear):
        prune.l1_unstructured(module, name='weight', amount=0.7)

print(dict(m2_copy.named_buffers()).keys())  # to verify that all masks exist

dict_keys(['features.conv0.weight_mask', 'features.norm0.running_mean', 'features.norm0.running_var', 'features.norm0.num_batches_tracked', 'features.denseblock1.denselayer1.norm1.running_mean', 'features.denseblock1.denselayer1.norm1.running_var', 'features.denseblock1.denselayer1.norm1.num_batches_tracked', 'features.denseblock1.denselayer1.conv1.weight_mask', 'features.denseblock1.denselayer1.norm2.running_mean', 'features.denseblock1.denselayer1.norm2.running_var', 'features.denseblock1.denselayer1.norm2.num_batches_tracked', 'features.denseblock1.denselayer1.conv2.weight_mask', 'features.denseblock1.denselayer2.norm1.running_mean', 'features.denseblock1.denselayer2.norm1.running_var', 'features.denseblock1.denselayer2.norm1.num_batches_tracked', 'features.denseblock1.denselayer2.conv1.weight_mask', 'features.denseblock1.denselayer2.norm2.running_mean', 'features.denseblock1.denselayer2.norm2.running_var', 'features.denseblock1.denselayer2.norm2.num_batches_tracked', 'features.dens

In [ ]:
#RANDOM TEST


bmodel.eval()

with torch.no_grad():
    n_correct = 0
    n_samples = 0
    n_class_correct = [0 for i in range(100)]
    n_class_samples = [0 for i in range(100)]
    for images, labels in val_loader:
        batch_size = images.size(0)
        images = images.to(device)
        labels = labels.to(device)
        outputs = bmodel(images)
        # max returns (value ,index)
        _, predicted = torch.max(outputs, 1)
        n_samples += labels.size(0)
        n_correct += (predicted == labels).sum().item()

        for i in range(batch_size):
            label = labels[i]
            pred = predicted[i]
            if (label == pred):
                n_class_correct[label] += 1
            n_class_samples[label] += 1

    acc = 100.0 * n_correct / n_samples
    print(f'Accuracy of the network: {acc} %')

    #for i in range(10):
        #acc = 100.0 * n_class_correct[i] / n_class_samples[i]
        #print(f'Accuracy of {classes[i]}: {acc} %')

Accuracy of the network: 0.93 %


In [ ]:
#TRAINING SUBNET1 MODEL WITH KD
class DistillKL(nn.Module):
    """Distilling the Knowledge in a Neural Network"""
    def __init__(self, T):
        super(DistillKL, self).__init__()
        self.T = T

    def forward(self, y_s, y_t):
        p_s = F.log_softmax(y_s/self.T, dim=1)
        p_t = F.softmax(y_t/self.T, dim=1)
        loss = F.kl_div(p_s, p_t, size_average=False) * (self.T**2) / y_s.shape[0]
        return loss

# Hyper-parameters

alpha = 0.95
num_epochs = 100
learning_rate = 0.01
criterionC = nn.CrossEntropyLoss()
optimizerC = torch.optim.SGD(m_copy.parameters(), momentum= 0.9, weight_decay=0.005, lr=learning_rate) #weight_decay=0.005
n_total_steps = len(train_loader)
scheduler3 = lr_scheduler.MultiStepLR(optimizerA, milestones=(30, 70), gamma=0.1)
#scheduler3 = lr_scheduler.StepLR(optimizerC, step_size=1, gamma=0.1)

for epoch in range(num_epochs): #I decided to train the model for 50 epochs
    # Decay Learning Rate
    scheduler3.step()

    loss_var = 0
    model_s = m_copy.train()
    model_t = bmodel.eval()
    cri_cls = criterionC
    cri_kd = DistillKL(T=5)
    for idx, (data, target) in enumerate(train_loader):
      data = data.to(device=device)
      target = target.to(device=device)
      #data, target = data.to(device), target.to(device)

      optimizerC.zero_grad()
      output_s = model_s(data).to(device)
      output_t = model_t(data).to(device)

      loss_cls = cri_cls(output_s, target)
      loss_kd = cri_kd(output_s, output_t)
      loss = loss_cls + loss_kd
      #loss = alpha*loss_cls + (1-alpha)*loss_kd
      loss.backward()

      optimizerC.step()
      loss_var += loss.item()
      if idx%64==0:
          print(f'Epoch [{epoch+1}/{num_epochs}] || Step [{idx+1}/{len(train_loader)}] || Loss:{loss_var/len(train_loader)}')
    print(f"Loss at epoch {epoch+1} || {loss_var/len(train_loader)}")

    with torch.no_grad():
        correct = 0
        samples = 0
        for idx, (data, target) in enumerate(val_loader):
            data = data.to(device=device)
            target = target.to(device=device)
            outputs = m_copy(data)
            _, preds = outputs.max(1)
            correct += (preds == target).sum()
            samples += preds.size(0)
        print(f"accuracy {float(correct) / float(samples) * 100:.2f} percentage || Correct {correct} out of {samples} samples")
    PATH3 = '/home/lm/m_copy_resnet34.pt'
    torch.save(m_copy.state_dict(), PATH3)
print('Finished Training')
#PATH3 = 'm_copy.pt'
#torch.save(m_copy.state_dict(), PATH3)

In [ ]:
#TRAINING SUBNET2 MODEL WITH KD
class DistillKL(nn.Module):
    """Distilling the Knowledge in a Neural Network"""
    def __init__(self, T):
        super(DistillKL, self).__init__()
        self.T = T

    def forward(self, y_s, y_t):
        p_s = F.log_softmax(y_s/self.T, dim=1)
        p_t = F.softmax(y_t/self.T, dim=1)
        loss = F.kl_div(p_s, p_t, size_average=False) * (self.T**2) / y_s.shape[0]
        return loss

# Hyper-parameters

alpha = 0.95
num_epochs = 100
learning_rate = 0.01
criterionC = nn.CrossEntropyLoss()
optimizerC = torch.optim.SGD(m2_copy.parameters(), momentum= 0.9, weight_decay=0.005, lr=learning_rate) #weight_decay=0.005
n_total_steps = len(train_loader)
scheduler3 = lr_scheduler.MultiStepLR(optimizerA, milestones=(30, 70), gamma=0.1)
#scheduler3 = lr_scheduler.StepLR(optimizerC, step_size=1, gamma=0.1)

for epoch in range(num_epochs): #I decided to train the model for 50 epochs
    # Decay Learning Rate
    scheduler3.step()

    loss_var = 0
    model_s = m2_copy.train()
    model_t = bmodel.eval()
    cri_cls = criterionC
    cri_kd = DistillKL(T=5)
    for idx, (data, target) in enumerate(train_loader):
      data = data.to(device=device)
      target = target.to(device=device)
      #data, target = data.to(device), target.to(device)

      optimizerC.zero_grad()
      output_s = model_s(data).to(device)
      output_t = model_t(data).to(device)

      loss_cls = cri_cls(output_s, target)
      loss_kd = cri_kd(output_s, output_t)
      loss = loss_cls + loss_kd
      #loss = alpha*loss_cls + (1-alpha)*loss_kd
      loss.backward()

      optimizerC.step()
      loss_var += loss.item()
      if idx%64==0:
          print(f'Epoch [{epoch+1}/{num_epochs}] || Step [{idx+1}/{len(train_loader)}] || Loss:{loss_var/len(train_loader)}')
    print(f"Loss at epoch {epoch+1} || {loss_var/len(train_loader)}")

    with torch.no_grad():
        correct = 0
        samples = 0
        for idx, (data, target) in enumerate(val_loader):
            data = data.to(device=device)
            target = target.to(device=device)
            outputs = m2_copy(data)
            _, preds = outputs.max(1)
            correct += (preds == target).sum()
            samples += preds.size(0)
        print(f"accuracy {float(correct) / float(samples) * 100:.2f} percentage || Correct {correct} out of {samples} samples")
    PATH1 = '/home/lm/m2_copy_resnet34.pt'
    torch.save(m2_copy.state_dict(), PATH3)
print('Finished Training')